<a href="https://colab.research.google.com/github/lekhanakcs24-source/AI-LAB/blob/main/KB%20%20QUERY%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from itertools import product
import re

# Convert logical operators into Python operators
def prepare_expression(expr):
    expr = expr.replace("IMPLIES", ">>")
    expr = expr.replace("AND", " and ")
    expr = expr.replace("OR", " or ")
    expr = expr.replace("NOT", " not ")

    # Convert implication A >> B into (not A or B)
    # For simplicity, enter implications as: (not A or B)
    if ">>" in expr:
        raise ValueError(
            "Write implication A -> B as (NOT A OR B)"
        )

    return expr


def evaluate(expr, model):
    expr = prepare_expression(expr)

    # Replace each variable with its truth value
    for symbol in sorted(model, key=len, reverse=True):
        expr = re.sub(
            r"\b" + re.escape(symbol) + r"\b",
            str(model[symbol]),
            expr
        )

    # Evaluate only expressions made of safe logical syntax
    if not re.fullmatch(r"[\w\s()]+", expr):
        raise ValueError("Invalid expression")

    return bool(eval(expr, {"__builtins__": {}}, {}))


def get_symbols(expressions):
    keywords = {"AND", "OR", "NOT", "IMPLIES"}
    symbols = set()

    for expr in expressions:
        words = re.findall(r"\b[A-Za-z]\w*\b", expr)
        symbols.update(
            word for word in words if word.upper() not in keywords
        )

    return sorted(symbols)


def main():
    print("TRUTH TABLE ENUMERATION")
    print("Operators: AND, OR, NOT")
    print("Use parentheses for grouping.")
    print("Example: (P AND Q)")
    print("Write implication P -> Q as (NOT P OR Q)\n")

    kb = input("Enter Knowledge Base (KB): ").upper()
    query = input("Enter Query: ").upper()

    expressions = [kb, query]
    symbols = get_symbols(expressions)

    if not symbols:
        print("Error: Enter at least one proposition.")
        return

    print("\nSymbols:", ", ".join(symbols))
    print("\nTruth Table:")
    print(" | ".join(symbols) + " | KB | Query")
    print("-" * (len(symbols) * 5 + 18))

    entails = True
    model_count = 0

    for values in product([True, False], repeat=len(symbols)):
        model = dict(zip(symbols, values))

        try:
            kb_value = evaluate(kb, model)
            query_value = evaluate(query, model)
        except Exception as error:
            print("Expression error:", error)
            return

        model_count += 1
        row = ["T" if model[s] else "F" for s in symbols]
        row += ["T" if kb_value else "F",
                "T" if query_value else "F"]

        print(" | ".join(row))

        if kb_value and not query_value:
            entails = False

    print("\nTotal models checked:", model_count)

    if entails:
        print("RESULT: KB entails the Query (KB |= Query)")
    else:
        print("RESULT: KB does NOT entail the Query (KB |/= Query)")


if __name__ == "__main__":
    main()


TRUTH TABLE ENUMERATION
Operators: AND, OR, NOT
Use parentheses for grouping.
Example: (P AND Q)
Write implication P -> Q as (NOT P OR Q)

Enter Knowledge Base (KB): P AND(NOT Q)
Enter Query: Q

Symbols: P, Q

Truth Table:
P | Q | KB | Query
----------------------------
T | T | F | T
T | F | T | F
F | T | F | T
F | F | F | F

Total models checked: 4
RESULT: KB does NOT entail the Query (KB |/= Query)
